# Legal & Compliance RAG System — Hedge Edge

This notebook demonstrates how to use the Legal Compliance skill's RAG system,
which integrates with Google NotebookLM for grounded, cited answers to legal questions.

## Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  Legal Compliance Skill                                     │
│  .agents/skills/legal-compliance/                           │
│                                                             │
│  ┌──────────────┐    ┌──────────────────┐                   │
│  │  execution/   │    │  resources/       │                  │
│  │              │    │                  │                    │
│  │  legal_query  │◄───│  uk-gdpr         │                  │
│  │  _engine.py   │    │  fca-promotions  │                  │
│  │              │    │  privacy-policy  │                    │
│  │  gdpr_checker │    │  terms-of-service│                  │
│  │  .py         │    │  risk-disclaimers│                   │
│  │              │    │  ib-checklist    │                    │
│  │  fca_auditor  │    │  dsar-process   │                   │
│  │  .py         │    │  regulatory-     │                   │
│  │              │    │   landscape      │                   │
│  │  setup_legal  │    │  data-processing│                   │
│  │  _notebook.py │    │   -register     │                   │
│  └──────┬───────┘    └─────────┬────────┘                   │
│         │                      │                             │
│         ▼                      ▼                             │
│  ┌──────────────────────────────────────┐                    │
│  │  NotebookLM  ("Hedge Edge Legal")    │                   │
│  │  RAG-powered Q&A with cited sources  │                   │
│  └──────────────────────────────────────┘                    │
└─────────────────────────────────────────────────────────────┘
```

## Prerequisites

```bash
pip install notebooklm-py[browser]
notebooklm login   # one-time Google auth
```

## 1. Setup — Path Configuration

First, configure paths to the skill directory and shared modules.

In [ ]:
import sys
import os
from pathlib import Path

# Navigate to the skill root
SKILL_DIR = Path(r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge\Orchestrator\Hedge Edge Business\IDE 1\agents\STRATEGY\.agents\skills\legal-compliance")
EXECUTION_DIR = SKILL_DIR / "execution"
RESOURCES_DIR = SKILL_DIR / "resources"
ORCHESTRATOR_ROOT = Path(r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge\Orchestrator")

# Add to path so we can import shared modules
sys.path.insert(0, str(ORCHESTRATOR_ROOT))
sys.path.insert(0, str(EXECUTION_DIR))

print(f"Skill directory:    {SKILL_DIR}")
print(f"Resources:          {RESOURCES_DIR}")
print(f"Execution scripts:  {EXECUTION_DIR}")
print(f"Orchestrator root:  {ORCHESTRATOR_ROOT}")

## 2. Check Resources

Verify all legal knowledge base documents exist in the `resources/` folder.

In [ ]:
# List all resource documents
print("Legal Knowledge Base Documents")
print("=" * 50)

if RESOURCES_DIR.exists():
    for f in sorted(RESOURCES_DIR.iterdir()):
        size = f.stat().st_size
        print(f"  {'✅' if size > 0 else '❌'} {f.name} ({size:,} bytes)")
else:
    print("  ❌ Resources directory not found!")

## 3. Setup NotebookLM

Create the "Hedge Edge Legal" notebook and load all resource documents.
This only needs to be run **once** (or when resources are updated).

In [ ]:
# Run the setup script to create the NotebookLM notebook
# This loads all resources into NotebookLM for RAG queries

from setup_legal_notebook import list_sources, setup
import asyncio

# First, check all sources exist
all_ok = list_sources()

if all_ok:
    print("\nAll sources found! Ready to create notebook.")
else:
    print("\nSome sources missing — notebook will be created with available sources.")

In [ ]:
# Uncomment to actually create the NotebookLM notebook:
# await setup()

# Or run from CLI:
# !python "{EXECUTION_DIR / 'setup_legal_notebook.py'}"

## 4. Query the RAG System

Ask legal questions and get grounded, cited answers from the NotebookLM notebook.

In [ ]:
from legal_query_engine import query_legal
import json

# Example: GDPR question
result = await query_legal(
    question="Does Hedge Edge need ICO registration?",
    jurisdiction="uk",
    doc_type="gdpr"
)

print(f"Risk Level: {result['risk_level']}")
print(f"NotebookLM used: {result['notebooklm_used']}")
print(f"\n{result['answer']}")

In [ ]:
# Example: FCA financial promotions question
result = await query_legal(
    question="Can we use the phrase '28x ROI' in our marketing?",
    jurisdiction="uk",
    doc_type="financial-promotions"
)

print(f"Risk Level: {result['risk_level']}")
print(f"\n{result['answer']}")

In [ ]:
# Example: IB agreement question
result = await query_legal(
    question="What do we need to disclose about our IB commissions?",
    jurisdiction="uk",
    doc_type="ib-agreement"
)

print(f"Risk Level: {result['risk_level']}")
print(f"\n{result['answer']}")

## 5. Run GDPR Compliance Audit

Check Hedge Edge's GDPR compliance status with a structured audit.

In [ ]:
from gdpr_compliance_checker import run_audit, print_audit

# Run a full GDPR compliance audit
results = run_audit("full")
print_audit(results)

In [ ]:
# Focus on data transfers only
transfer_results = run_audit("data-transfers")
print_audit(transfer_results)

## 6. Scan Marketing Copy for FCA Violations

Check marketing text for prohibited financial promotions language.

In [ ]:
from financial_promotions_auditor import scan_text, print_scan_results, REQUIRED_DISCLAIMERS

# Test some marketing copy
sample_copy = """
Hedge Edge guarantees your prop firm challenges are risk-free.
With our 28x ROI, you'll always make money while you sleep.
Our financial advice is to start hedging today for guaranteed profits.
"""

violations = scan_text(sample_copy)
print_scan_results(violations, "Sample marketing copy")

In [ ]:
# Scan the actual landing page content
landing_page = Path(r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge\Current landing page\src\data\siteData.ts")

if landing_page.exists():
    content = landing_page.read_text(encoding="utf-8")
    violations = scan_text(content)
    print_scan_results(violations, str(landing_page))
else:
    print("Landing page data file not found")

## 7. View Required Disclaimers

Get the approved disclaimer language for different marketing materials.

In [ ]:
print("APPROVED DISCLAIMERS FOR HEDGE EDGE")
print("=" * 50)

for key, disc in REQUIRED_DISCLAIMERS.items():
    print(f"\n--- {key.upper()} ---")
    print(f"Where: {disc['where']}")
    print(f"Rule:  {disc['rule']}")
    print(f"Text:  {disc['text']}")

## 8. Check NotebookLM Budget

Monitor daily query budget usage.

In [ ]:
from shared.notebooklm_client import budget_remaining

remaining = budget_remaining()
print(f"NotebookLM daily budget remaining: {remaining} queries")

## CLI Quick Reference

All execution scripts can also be run from the terminal:

```bash
# Setup NotebookLM notebook
python execution/setup_legal_notebook.py
python execution/setup_legal_notebook.py --list

# Query the RAG system
python execution/legal_query_engine.py -q "Do we need ICO registration?"
python execution/legal_query_engine.py -q "GDPR for Supabase" -j uk -d gdpr
python execution/legal_query_engine.py --budget

# GDPR audit
python execution/gdpr_compliance_checker.py --audit full
python execution/gdpr_compliance_checker.py --audit data-transfers --json

# FCA promotions scan
python execution/financial_promotions_auditor.py -t "Your marketing text here"
python execution/financial_promotions_auditor.py -f path/to/marketing_copy.md
python execution/financial_promotions_auditor.py --generate-disclaimers
```

## Task Dispatcher

Tasks are also registered in the Business Strategist `run.py`:

```bash
python run.py --task legal-query --action ask --question "GDPR question"
python run.py --task gdpr-audit --action full
python run.py --task fca-scan --action scan-landing-page
python run.py --task legal-setup --action create-notebook
```